# Notebook 10 — Learned geometry as a JEPA encoder

**Run All launches the real experiment in detached tmux.** The worker continues independently of the browser. Expected runtime is approximately **1–2 hours**, subject to actual calibration; the fixed **three-hour cap** includes preflight, training and evaluation.

The question is whether learned patch geometry improves the complete custom JEPA pipeline. Kernel sums are evaluated directly. **This is not an FMM implementation or speedup claim.** Coordinate maps are learned embeddings, not proved manifold charts.

| Encoder | Interaction geometry | Evaluation |
|---|---|---|
| ViT | Learned dot-product attention | Exact attention |
| Spatial kernel | Fixed image positions, learned kernel scales | Dense normalized smooth kernel |
| Learned kernel | Separate learned query/source coordinates with spatial anchors | Dense normalized smooth kernel |

All three use 128px images, 8px patches, width 192, four blocks, three heads, identical 4× expansion pointwise MLPs and the unchanged JEPA predictor. Common stem, MLPs, norms and predictor initialization is copied with independent storage. Kernel controls share common value/output and kernel parameters. Active counts differ and are reported.

**Six training arms:** seeds 20261301/20261302, three architectures, 350 successful updates each at effective and physical batch 128. Paired arms receive identical samples, augmentations and masks. Initial/final frozen EMA encoders each receive fixed 20 epoch linear probes. Two seeds are exploratory; a higher mean does not establish statistical significance.


## Kernel encoder mathematics

For normalized patch features $u_i$ and position $r_i$, learn low-dimensional coordinates $a_i=\phi_Q(u_i,r_i)$ and $b_i=\phi_K(u_i,r_i)$, and values $v_i=Vu_i$. The spatial control uses the same coordinate space without content displacement. A positive mixture of smooth radial kernels defines

$$m_i=\frac{\sum_{j\in C}K_\theta(a_i,b_j)v_j}{\sum_{j\in C}K_\theta(a_i,b_j)}.$$

Only valid visible context patches enter a context encoder. Each block applies an output projection, residual connection and pointwise MLP. Both online and EMA target encoders use the selected architecture; predictor, target normalization and loss remain unchanged.

FMM would later approximate these same sums. Dense evaluation isolates the learning hypothesis. A useful kernel still needs a separate error-controlled acceleration and scaling test before an FMM claim.

The implementation evaluates this positive-kernel ratio through log-sum-exp and row softmax, without an added epsilon; every sample must contain a valid source.


## Inputs, outputs and exact resume

Inputs: sealed Food101 10,100 training/probe-training images and 2,020 development images. Official test reserved. Mask/optimizer/probe recipe comes from the hash-bound previous registered configuration.

Outputs: artifacts/notebook_10/manifold-kernel-pilot-001/; report: stages/learning/stage_results.md. Full-state checkpoints: checkpoints/manifold-kernel-pilot-001/. Save each epoch/stage boundary, every 50 updates or 15 minutes; latest two and best retained, with atomic writes and checksum-verified manifests. One writer per run. Workspace is on persistent boot disk; instance-deletion policy remains unverified. Copy final/best checkpoints off-instance and verify hashes before deleting the instance.

Before training, the worker verifies actual checkpoint creation and bitwise four-step versus interrupted two-plus-two-step recovery for all architectures, then checks encoder/probe integration. Calibration weights are discarded; quality outcomes never alter budgets.

**Resume:** choose the ahr-go kernel and Run All unchanged with the same RUN_ID before the original deadline. It reconnects to an active worker or resumes valid states at their exact step. Invalid manifests fail explicitly. After completion, Run All shows results without retraining. The original clock never resets.

**Session:** fmm-manifold-kernel-manifold-kernel-pilot-001. Attach: tmux attach -t fmm-manifold-kernel-manifold-kernel-pilot-001. Detach: Ctrl-b, then d. An independent deadline watcher has a separate session. GPU 0 is used; the separate AHR session on GPU 1 is preserved.


**Evaluation recovery:** training completed all six arms. A device mismatch in the descriptive geometry check was repaired without changing training, cached features, probes or the original deadline. For an interrupted evaluation, resume with [10_Evaluation_Recovery.ipynb](../recoveries/10_Evaluation_Recovery.ipynb) and Run All before 7 October 2026 at 00:59:58 Europe/Dublin. The same detached worker/session and clock are used. After completion this notebook displays the original report normally.


## 1. Load tagged definitions

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import nbformat, importlib.util, sys
from IPython.display import display, Markdown, Image
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)
_spec = importlib.util.spec_from_loader("fmm10_workflow", _boot["TaggedNotebookLoader"](ROOT / "experiments/10_manifold_kernel/workflow.ipynb"))
WORKFLOW = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = WORKFLOW
_spec.loader.exec_module(WORKFLOW)


## 2. Fixed configuration
Three variants, two fresh seeds, 350 updates, 20 epoch initial/final probes and a three-hour cap are fixed before outcomes.

In [ ]:
RUN_ID = "manifold-kernel-pilot-001"
CONFIG = WORKFLOW.study_configuration(RUN_ID)
CONFIG


## 3. Check readiness and model sizes
This check creates no experiment clock, checkpoints or worker. It verifies the saved build-validation certificates and reports actual architecture counts. Scientific results remain NOT_RUN until the experiment completes.

In [ ]:
READINESS = WORKFLOW.readiness(ROOT, CONFIG)
import pandas as pd
display(pd.DataFrame(READINESS["architecture_summary"]))
display({key: READINESS[key] for key in ("status", "production_run_started", "expected_runtime", "planned_training_runs", "planned_probe_evaluations", "checkpoint_location", "scientific_results")})


## 4. Launch or reconnect
**This cell starts the real experiment.** Run All executes it automatically. The saved binding fixes the original clock; the detached worker performs calibration and recovery checks before training.

In [ ]:
LAUNCH = WORKFLOW.launch_study(ROOT, CONFIG)
DISPLAY_LAUNCH = {key: value for key, value in LAUNCH.items() if key != "command"}
if LAUNCH.get("deadline_unix"):
    DISPLAY_LAUNCH["hard_stop_dublin"] = datetime.fromtimestamp(LAUNCH["deadline_unix"], ZoneInfo("Europe/Dublin")).isoformat()
display(DISPLAY_LAUNCH)


## 5. Inspect progress
Rerun this cell for stage status, saved update counts and time remaining. Accuracy comparisons appear after all training runs and evaluations complete.

In [ ]:
STATUS = WORKFLOW.inspect_study(ROOT, RUN_ID)
PROGRESS = STATUS.get("saved_training_progress", [])
display({key: value for key, value in STATUS.items() if key != "saved_training_progress"})
if PROGRESS:
    display(pd.DataFrame(PROGRESS))


## 6. Read completed results
Only real completed experiment results are rendered here. A partial run is not analyzed as the planned full study.

In [ ]:
REPORT_FOLDER = ROOT / "artifacts/notebook_10" / RUN_ID / "stages/learning"
REPORT = REPORT_FOLDER / "stage_results.md"
if STATUS.get("status") == "COMPLETED" and REPORT.exists():
    display(Markdown("\n".join(line for line in REPORT.read_text().splitlines() if not line.startswith("!["))))
    for figure in ("stage_quality.png", "stage_compute.png", "stage_contrasts.png"):
        figure_path = REPORT_FOLDER / figure
        if figure_path.exists():
            display(Image(filename=str(figure_path)))
else:
    display(Markdown("**No complete scientific result yet.** Use the progress cell above; partial or validation-only scores are not pilot conclusions."))


## Interpretation after completion

Read own-initialization gains alongside final development accuracy. Compare learned minus spatial for content geometry, and both kernels minus ViT for encoder replacement. Review measured time/peak memory and individual seeds. Two pairs on reused development membership cannot establish superiority or equivalence. No automatic larger run.

Coordinates are not proved manifold charts; direct sums retain quadratic cost. FMM approximation is a subsequent engineering hypothesis.

Method references: [I-JEPA](https://arxiv.org/abs/2301.08243), [Multipole Graph Neural Operator](https://proceedings.neurips.cc/paper_files/paper/2020/file/4b21cf96d4cf612f239a6c322b10c8fe-Paper.pdf), [Fast Multipole Attention](https://arxiv.org/abs/2310.11960). Controlled small custom JEPA implementation, not a reproduction of published large-scale scores.
